In [100]:
# Inicializa o histórico vazio
historico = []

In [101]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [102]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [103]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [104]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Removemos o parâmetro 'language' fixo para permitir a detecção automática
result = model.transcribe(record_file, fp16=False)
transcription = result["text"]
idioma_detectado = result["language"]  # O Whisper nos diz qual língua ouviu!

print(f"O Whisper detectou o idioma: {idioma_detectado}")
print(f"Usuário disse: {transcription}")

O Whisper detectou o idioma: pt
Usuário disse:  O que eu posso aprender com ele?


# 3. Integração com a API do Gemini AI


In [105]:
!pip install -q -U google-generativeai

In [106]:
#!pip install openai==0.28

In [107]:
import google.generativeai as genai

genai.configure(api_key="SUA_CHAVE_AQUI")
model_gemini = genai.GenerativeModel('gemini-2.5-flash')

# 1. Adicionamos a pergunta atual ao histórico
historico.append({"role": "user", "parts": [transcription]})

# 2. Criamos um prompt que pede para a IA respeitar o idioma e o contexto
instrucao_sistema = f"Responda de forma curta e natural no idioma {idioma_detectado}."

try:
    # 3. Enviamos o histórico completo para o Gemini
    # O Gemini usa o método start_chat para gerenciar memória facilmente
    chat = model_gemini.start_chat(history=historico[:-1]) # Enviamos o passado
    response = chat.send_message(f"{instrucao_sistema} Pergunta: {transcription}")

    gemini_response = response.text

    # 4. Salvamos a resposta da IA no histórico para a próxima rodada
    historico.append({"role": "model", "parts": [gemini_response]})

    print(f"Gemini respondeu: {gemini_response}")

except Exception as e:
    print(f"Erro ao gerar resposta: {e}")

Gemini respondeu: Depende do que 'ele' é! 😉


# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [108]:
!pip install gTTS

In [109]:
from gtts import  gTTS

# Usamos idioma_detectado (que o Whisper descobriu)
# em vez da variável fixa language
gtts_object = gTTS(text=gemini_response, lang=idioma_detectado, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))

In [110]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem